# LangChain LLM Chain

## 1. Prerequisites and imports
First, the necessary environment variables and tools are loaded.

In [ ]:
pip install python-dotenv

In [ ]:
pip install -U langchain-google-genai langgraph

In [ ]:
import os
from dataclasses import dataclass
from dotenv import load_dotenv

load_dotenv(dotenv_path="key.env")

True

## 2. Define the system prompt
Here you define the agent's behavior rules.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.structured_output import ToolStrategy

SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:
- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. 
If you can tell from the question that they mean wherever they are, 
use the get_user_location tool to find their location."""

## 3. Create tools
The functions that the agent can perform are defined. Note the use of Context to simulate user data.

In [7]:
# --- CONTEXT SCHEMA ---
@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

In [8]:
# --- TOOLS ---
@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    # Mock database: User 1 is in Florida, others in SF
    return "Florida" if user_id == "1" else "SF"

## 4. Configure model
Gemini is configured to be the "brain" of the agent.

In [26]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

## 5. Define response format
Create the data structure that the agent should always return.

In [19]:
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    punny_response: str
    weather_conditions: str | None = None

## 6. Add memory
This configures message saving to RAM for this session.

In [20]:
checkpointer = InMemorySaver()

## 7. Create and run the agent
Finally, everything is assembled and the interaction tests are run.

In [27]:
# Assemble the agent
agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)

# Configuration with a unique thread_id for conversation state
config = {"configurable": {"thread_id": "1"}}

# Run 1: Asking about weather (The agent will use get_user_location)
response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")
)
print("First Response:")
print(response['structured_response'])

# Run 2: Continuing the conversation (The agent uses memory)
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)
print("\nSecond Response (Memory):")
print(response['structured_response'])

First Response:
ResponseFormat(punny_response='Looks like the weather in Florida is quite a *bright* spot! No need to *frown* upon these conditions!', weather_conditions="It's always sunny in Florida!")

Second Response (Memory):
ResponseFormat(punny_response="You're welcome! I'm always here to *weather* any questions you have!", weather_conditions=None)


## 8. Example test

In [29]:
# TEST
config_user2 = {"configurable": {"thread_id": "2"}}

print("--- TESTING USER 2 (San Francisco) ---")
# Run 1: Asking about weather
response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config_user2,
    context=Context(user_id="2")
)
print("First Response (User 2):")
print(response['structured_response'])

# Run 2: Continuing the conversation
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config_user2,
    context=Context(user_id="2")
)
print("\nSecond Response (User 2 - Memory):")
print(response['structured_response'])

--- TESTING USER 2 (San Francisco) ---
First Response (User 2):
ResponseFormat(punny_response="Don't let anyone rain on your parade, because it's always sunny in SF!", weather_conditions='Sunny')

Second Response (User 2 - Memory):
ResponseFormat(punny_response="You're welcome! I'm always here to weather any storm of questions you might have.", weather_conditions=None)
